# Analyse des Satz-Klassifikators (BiLSTM)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook evaluiert den trainierten **Satz-Klassifikator** (AS vs. LS) auf Satzebene:
1. Laden des Modells und der Gewichte.
2. Rekonstruktion/Laden des Vokabulars.
3. Evaluierung auf Test- und Validierungsdaten.
4. Interaktive Klassifikation eigener Sätze.


In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Arbeitsverzeichnis auf Projekt-Root setzen
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


## 1. Konfiguration & Hyperparameter
Bitte passe die Werte so an, wie sie beim Training von `1_binary_train_sentence_model.py` verwendet wurden.


In [ ]:
CSV_PATH = "data/analysis/corpus_master.csv"
MODEL_PATH = "results/models/best_model_sim_0.8_0.98.pt"  # oder passender Pfad
MIN_SIM = 0.8
MAX_SIM = 0.98
MIN_SENT_LEN = 3
MAX_SEQ_LEN = 100
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


## 2. Daten laden & Vokabular rekonstruieren
Da die Vokabular-Klasse dynamisch zur Trainingszeit aufgebaut wurde, rekonstruieren wir sie hier unter Verwendung desselben Seeds und Splits.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

class Vocab:
    def __init__(self, sentences, max_size=20000, min_freq=2):
        counter = Counter()
        for sent in sentences:
            counter.update(sent)
        self.itos = ["<pad>", "<unk>"]
        self.stoi = {"<pad>": 0, "<unk>": 1}
        for token, freq in counter.most_common(max_size):
            if freq >= min_freq:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)
    def __len__(self): return len(self.itos)
    def encode(self, tokens):
        return [self.stoi.get(t, self.stoi["<unk>"]) for t in tokens]

print("Lade Datensatz und segmentiere in Sätze (dauert kurz)...")
df = pd.read_csv(CSV_PATH)
mask = (df["semantic_similarity_8192"] >= MIN_SIM) & (df["semantic_similarity_8192"] <= MAX_SIM)
df_filtered = df[mask]

ls_sentences = []
as_sentences = []
nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    for sent in nlp(str(row["ls_text"])).sents:
        tokens = [t.text.lower() for t in sent if not t.is_space]
        if len(tokens) >= MIN_SENT_LEN: ls_sentences.append(tokens)
    for sent in nlp(str(row["as_text"])).sents:
        tokens = [t.text.lower() for t in sent if not t.is_space]
        if len(tokens) >= MIN_SENT_LEN: as_sentences.append(tokens)

min_len = min(len(ls_sentences), len(as_sentences))
random.shuffle(ls_sentences)
random.shuffle(as_sentences)
ls_sentences = ls_sentences[:min_len]
as_sentences = as_sentences[:min_len]

X = ls_sentences + as_sentences
y = [1] * len(ls_sentences) + [0] * len(as_sentences)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.11, random_state=42, stratify=y_train_val)

vocab = Vocab(X_train)
print(f"Rekonstruierte Vokabular-Größe: {len(vocab)}")
print(f"Test-Sätze: {len(X_test)}")


## 3. Modell definieren & laden


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=1):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden))

model = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Modell-Gewichte geladen!")
else:
    print(f"FEHLER: Modellpfad {MODEL_PATH} existiert nicht!")
model.eval()


## 4. Evaluierung auf den Testdaten


In [ ]:
all_preds = []
all_targets = []

with torch.no_grad():
    for x_tokens, label in zip(X_test, y_test):
        encoded = vocab.encode(x_tokens)[:MAX_SEQ_LEN]
        padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
        inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
        
        logits = model(inp)
        prob = torch.sigmoid(logits).item()
        pred = 1 if prob >= 0.5 else 0
        
        all_preds.append(pred)
        all_targets.append(label)

print("--- Klassifikationsbericht ---")
print(classification_report(all_targets, all_preds, target_names=["AS (Alltagssprache)", "LS (Leichte Sprache)"]))
print("Accuracy:", accuracy_score(all_targets, all_preds))
print("Balanced Accuracy:", balanced_accuracy_score(all_targets, all_preds))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["AS", "LS"], yticklabels=["AS", "LS"])
plt.title("Confusion Matrix (Testset)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()


## 5. Eigene Sätze testen
Gib hier beliebige deutsche Sätze ein, um zu prüfen, ob das Modell sie als Alltagssprache (AS, nahe bei 0.0) oder Leichte Sprache (LS, nahe bei 1.0) klassifiziert.


In [ ]:
def predict_sentence(text):
    # Tokenisierung mit SpaCy
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = vocab.encode(tokens)[:MAX_SEQ_LEN]
    padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
    inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        logits = model(inp)
        prob = torch.sigmoid(logits).item()
    
    label = "Leichte Sprache (LS)" if prob >= 0.5 else "Alltagssprache (AS)"
    print(f"Text: {text}")
    print(f"-> Wahrscheinlichkeit für Leichte Sprache: {prob:.4f}")
    print(f"-> Klasse: {label}\n")

# Testen mit Beispielen
predict_sentence("Wir möchten Ihnen helfen, die Anträge richtig auszufüllen.")
predict_sentence("Es ist von fundamentaler Bedeutung, die legislativen Rahmenbedingungen präzise zu analysieren.")
predict_sentence("Das ist ein einfacher Text in leichter Sprache.")


In [ ]:
# Interaktive Zelle
custom_text = "Schreiben Sie hier Ihren eigenen Satz..."
predict_sentence(custom_text)
